# Phase 3: Feature Engineering

This notebook covers feature engineering for the three unified datasets. We will:
1. **Parse dates:** Extract temporal features (`year`, `month`, `day_of_week`).
2. **Handle target scaling:** Log-transform right-skewed target variables (like microplastic concentration).
3. **Encode categorical fields:** Fit `OneHotEncoder` for low-cardinality classes and save the fitted state.
4. **Scale numerical features:** Fit `StandardScaler` to put features on similar scales and save the state.
5. **Sanitize column names:** Replace `[`, `]`, and `<` to prevent XGBoost execution errors.
6. **Split & Save:** Partition the engineered datasets into Train (80%) and Test (20%) and export them to `data/processed/`.

## 1. Setup Environment & Load Cleaned Data

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data_dir = "../data/"
processed_dir = "../data/processed/"
os.makedirs(processed_dir, exist_ok=True)

df_micro = pd.read_csv(os.path.join(data_dir, "cleaned_microplastics.csv"))
df_waste = pd.read_csv(os.path.join(data_dir, "cleaned_waste_events.csv"))
df_green = pd.read_csv(os.path.join(data_dir, "cleaned_green_initiatives.csv"))

def sanitize_cols(df):
    # Replace characters that cause issues with XGBoost column names
    df.columns = [
        str(col).replace('[', '(').replace(']', ')').replace('<', 'less_than') 
        for col in df.columns
    ]
    return df

print("Datasets loaded successfully.")

Datasets loaded successfully.


## 2. Preprocessing & Feature Engineering: Pillar 1 (Microplastics)

We will extract datetime features, scale numerical features, one-hot encode categoricals, log-transform the target, and save the fitted scalers/encoders.

In [2]:
# 1. Parse date
df_micro["collection_date"] = pd.to_datetime(df_micro["collection_date"])
df_micro["year"] = df_micro["collection_date"].dt.year
df_micro["month"] = df_micro["collection_date"].dt.month
df_micro["day_of_week"] = df_micro["collection_date"].dt.dayofweek

# Define column groups
num_cols = [
    "size_um", "sample_volume_l", "sample_weight_mg", "depth_m", "ph_level",
    "turbidity_ntu", "salinity_ppt", "elevation_m", "distance_to_coast_km",
    "area_km2", "annual_rainfall_mm", "avg_temperature_c", "pollution_index",
    "biodiversity_score", "year", "month", "day_of_week"
]
cat_cols = [
    "plastic_type", "shape", "color", "size_class", "collection_method",
    "lab_analysis_method", "location_type", "biome", "water_body_type",
    "urbanization_level", "season"
]
target_col = "concentration_particles_per_l"

# 2. Split data first to prevent data leakage during fitting
train_df, test_df = train_test_split(df_micro, test_size=0.2, random_state=42)

# 3. Fit standard scaler on numericals
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[num_cols])
test_scaled = scaler.transform(test_df[num_cols])

# 4. Fit one-hot encoder on categoricals
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
train_encoded = encoder.fit_transform(train_df[cat_cols])
test_encoded = encoder.transform(test_df[cat_cols])

# Convert to DataFrames
encoded_col_names = encoder.get_feature_names_out(cat_cols)

X_train_num = pd.DataFrame(train_scaled, columns=num_cols, index=train_df.index)
X_train_cat = pd.DataFrame(train_encoded, columns=encoded_col_names, index=train_df.index)
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
y_train = np.log1p(train_df[target_col]) # Log transform target

X_test_num = pd.DataFrame(test_scaled, columns=num_cols, index=test_df.index)
X_test_cat = pd.DataFrame(test_encoded, columns=encoded_col_names, index=test_df.index)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)
y_test = np.log1p(test_df[target_col])

# Sanitize column names for XGBoost
X_train = sanitize_cols(X_train)
X_test = sanitize_cols(X_test)

# Save train/test datasets
X_train.to_csv(os.path.join(processed_dir, "X_train_micro.csv"), index=False)
X_test.to_csv(os.path.join(processed_dir, "X_test_micro.csv"), index=False)
y_train.to_csv(os.path.join(processed_dir, "y_train_micro.csv"), index=False)
y_test.to_csv(os.path.join(processed_dir, "y_test_micro.csv"), index=False)

# Save preprocessors
os.makedirs("../models", exist_ok=True)
with open("../models/scaler_micro.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("../models/encoder_micro.pkl", "wb") as f:
    pickle.dump(encoder, f)

print(f"Pillar 1 preprocessed: X_train shape={X_train.shape}, X_test shape={X_test.shape}")

Pillar 1 preprocessed: X_train shape=(6800, 94), X_test shape=(1700, 94)


## 3. Preprocessing & Feature Engineering: Pillar 2 (Waste Cleanup Yield)

We will preprocess the waste events dataset. The target variable is `weight_collected_ton`.

In [3]:
# 1. Parse date
df_waste["event_date"] = pd.to_datetime(df_waste["event_date"])
df_waste["year"] = df_waste["event_date"].dt.year
df_waste["month"] = df_waste["event_date"].dt.month
df_waste["day_of_week"] = df_waste["event_date"].dt.dayofweek

# Define column groups
num_cols = [
    "volume_collected_m3", "microplastic_fraction_pct", "duration_hours", 
    "num_volunteers", "cost_usd", "co2_saved_kg", "num_staff", 
    "annual_collection_capacity_ton", "avg_cost_per_ton_usd", "recycling_rate_pct", 
    "partnership_count", "carbon_offset_ton_yr", "social_impact_score",
    "year", "month", "day_of_week"
]
cat_cols = [
    "waste_source", "plastic_grade", "disposal_mode", "transport_mode", 
    "weather_condition", "collector_type", "region", "fleet_type", 
    "focus_area", "certification", "funding_source"
]
target_col = "weight_collected_ton"

# 2. Split data
train_df, test_df = train_test_split(df_waste, test_size=0.2, random_state=42)

# 3. Fit scaler
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[num_cols])
test_scaled = scaler.transform(test_df[num_cols])

# 4. Fit encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
train_encoded = encoder.fit_transform(train_df[cat_cols])
test_encoded = encoder.transform(test_df[cat_cols])

# Convert to DataFrames
encoded_col_names = encoder.get_feature_names_out(cat_cols)

X_train_num = pd.DataFrame(train_scaled, columns=num_cols, index=train_df.index)
X_train_cat = pd.DataFrame(train_encoded, columns=encoded_col_names, index=train_df.index)
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
y_train = np.log1p(train_df[target_col]) # Log transform skew target

X_test_num = pd.DataFrame(test_scaled, columns=num_cols, index=test_df.index)
X_test_cat = pd.DataFrame(test_encoded, columns=encoded_col_names, index=test_df.index)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)
y_test = np.log1p(test_df[target_col])

# Sanitize column names for XGBoost
X_train = sanitize_cols(X_train)
X_test = sanitize_cols(X_test)

# Save datasets
X_train.to_csv(os.path.join(processed_dir, "X_train_waste.csv"), index=False)
X_test.to_csv(os.path.join(processed_dir, "X_test_waste.csv"), index=False)
y_train.to_csv(os.path.join(processed_dir, "y_train_waste.csv"), index=False)
y_test.to_csv(os.path.join(processed_dir, "y_test_waste.csv"), index=False)

# Save preprocessors
with open("../models/scaler_waste.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("../models/encoder_waste.pkl", "wb") as f:
    pickle.dump(encoder, f)

print(f"Pillar 2 preprocessed: X_train shape={X_train.shape}, X_test shape={X_test.shape}")

Pillar 2 preprocessed: X_train shape=(6800, 95), X_test shape=(1700, 95)


## 4. Preprocessing & Feature Engineering: Pillar 3 (Policy Impact Success Classifier)

We will define the target `achieved_policy_impact` as a binary target (1 for active policy success outcomes, 0 for None).

In [4]:
# 1. Parse date and calculate project duration
df_green["start_date"] = pd.to_datetime(df_green["start_date"])
df_green["end_date"] = pd.to_datetime(df_green["end_date"])
df_green["duration_days"] = (df_green["end_date"] - df_green["start_date"]).dt.days

df_green["start_year"] = df_green["start_date"].dt.year
df_green["start_month"] = df_green["start_date"].dt.month

# 2. Engineer target variable: 1 if it is an actual policy success outcome, 0 if None/No Policy Change
df_green["achieved_policy_impact"] = df_green["policy_outcome"].apply(
    lambda x: 1 if x in ["Minor Amendment", "Major Bill Passed", "Regulation Enacted"] else 0
)

# Define column groups
num_cols = [
    "total_budget_usd", "plastic_removed_ton", "co2_equivalent_saved_ton",
    "beneficiaries_reached", "media_coverage_count", "annual_budget_usd", 
    "num_projects_active", "num_countries_operating", "transparency_rating", 
    "media_presence_score", "co2_reduction_target_ton_yr", 
    "plastic_reduction_target_ton_yr", "policy_influence_score", 
    "duration_days", "start_year", "start_month"
]
cat_cols = [
    "initiative_type", "target_plastic_type", "geographic_scope", 
    "fund_source", "partner_sector", "data_quality_flag", "org_type", 
    "mandate", "primary_sdg_focus", "continent", "org_size_class"
]
target_col = "achieved_policy_impact"

# 3. Split data
train_df, test_df = train_test_split(df_green, test_size=0.2, random_state=42)

# 4. Fit scaler
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[num_cols])
test_scaled = scaler.transform(test_df[num_cols])

# 5. Fit encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
train_encoded = encoder.fit_transform(train_df[cat_cols])
test_encoded = encoder.transform(test_df[cat_cols])

# Convert to DataFrames
encoded_col_names = encoder.get_feature_names_out(cat_cols)

X_train_num = pd.DataFrame(train_scaled, columns=num_cols, index=train_df.index)
X_train_cat = pd.DataFrame(train_encoded, columns=encoded_col_names, index=train_df.index)
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
y_train = train_df[target_col]

X_test_num = pd.DataFrame(test_scaled, columns=num_cols, index=test_df.index)
X_test_cat = pd.DataFrame(test_encoded, columns=encoded_col_names, index=test_df.index)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)
y_test = test_df[target_col]

# Sanitize column names for XGBoost
X_train = sanitize_cols(X_train)
X_test = sanitize_cols(X_test)

# Save datasets
X_train.to_csv(os.path.join(processed_dir, "X_train_green.csv"), index=False)
X_test.to_csv(os.path.join(processed_dir, "X_test_green.csv"), index=False)
y_train.to_csv(os.path.join(processed_dir, "y_train_green.csv"), index=False)
y_test.to_csv(os.path.join(processed_dir, "y_test_green.csv"), index=False)

# Save preprocessors
with open("../models/scaler_green.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("../models/encoder_green.pkl", "wb") as f:
    pickle.dump(encoder, f)

print(f"Pillar 3 preprocessed: X_train shape={X_train.shape}, X_test shape={X_test.shape}")

Pillar 3 preprocessed: X_train shape=(6800, 95), X_test shape=(1700, 95)
